In [1]:
%pip install -U mp-api pandas pymatgen tqdm monty numpy

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json
import time
import getpass
from pathlib import Path
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from pymatgen.core import Composition
from monty.json import MontyEncoder
from mp_api.client import MPRester


BASE_DIR = Path("mp_sodium_battery_dataset")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
META_DIR = BASE_DIR / "metadata"
CORRECTED_DIR = BASE_DIR / "corrected"
FINAL_DIR = BASE_DIR / "final"

for folder in [RAW_DIR, PROCESSED_DIR, META_DIR, CORRECTED_DIR, FINAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR.resolve())

Base folder: C:\Users\IICT-1\next sustainability\mp_sodium_battery_dataset


In [3]:
MP_API_KEY = os.environ.get("MP_API_KEY")

if not MP_API_KEY:
    MP_API_KEY = getpass.getpass("Paste your Materials Project API key: ").strip()

if not MP_API_KEY:
    raise ValueError("No Materials Project API key provided.")

os.environ["MP_API_KEY"] = MP_API_KEY

print("Materials Project API key loaded successfully.")

Paste your Materials Project API key:  ········


Materials Project API key loaded successfully.


In [4]:
def make_jsonable(obj: Any) -> Any:
    return json.loads(json.dumps(obj, cls=MontyEncoder, default=str))


def doc_to_dict(doc: Any) -> dict:
    if isinstance(doc, dict):
        return make_jsonable(doc)

    if hasattr(doc, "model_dump"):
        try:
            return make_jsonable(doc.model_dump(mode="json"))
        except TypeError:
            return make_jsonable(doc.model_dump())

    if hasattr(doc, "dict"):
        return make_jsonable(doc.dict())

    return make_jsonable(dict(doc))


def csv_safe_value(x):
    if isinstance(x, (dict, list, tuple, set)):
        return json.dumps(make_jsonable(x), ensure_ascii=False)
    return x


def make_csv_safe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].map(csv_safe_value)
    return df


def save_records_as_json_and_csv(records: list[dict], stem: str):
    json_path = RAW_DIR / f"{stem}.json"
    csv_path = PROCESSED_DIR / f"{stem}.csv"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    df = pd.json_normalize(records, sep="__")
    df = make_csv_safe(df)
    df.to_csv(csv_path, index=False)

    print(f"Saved JSON: {json_path}")
    print(f"Saved CSV : {csv_path}")
    print(f"Shape     : {df.shape}")

    return df


def elements_from_formula(formula):
    if pd.isna(formula):
        return set()

    try:
        comp = Composition(str(formula))
        return set(el.symbol for el in comp.elements)
    except Exception:
        return set()


def elements_from_chemsys(chemsys):
    if pd.isna(chemsys):
        return set()

    return set(str(chemsys).split("-"))


def element_set_from_string(x):
    if pd.isna(x):
        return set()

    return set([e.strip() for e in str(x).split(",") if e.strip()])

In [5]:
TRANSITION_METALS = {
    "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu",
    "Zn", "Mo", "W"
}

HARD_EXCLUDE_ELEMENTS = {
    "Li", "Co", "Ni",
    "Cd", "Hg", "Pb", "As", "Tl", "U", "Th", "Pu", "Be",
    "Ag", "Au", "Pt", "Pd", "Ir", "Rh", "Ru", "Os", "Re"
}

SOFT_PENALTY_ELEMENTS = {
    "V", "Cr", "Mo", "W", "Sb", "Bi", "Se", "Te",
    "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb",
    "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Y"
}

HARD_EXCLUDE_V2 = {
    "Li", "Co", "Ni",
    "Ag", "Au", "Pt", "Pd", "Ir", "Rh", "Ru", "Os", "Re",
    "Cd", "Hg", "Pb", "As", "Tl", "U", "Th", "Pu", "Be",
    "H"
}

SOFT_PENALTY_V2 = {
    "V", "Cr", "Mo", "W", "Sb", "Bi", "Se", "Te",
    "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb",
    "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Y"
}

ALLOWED_FAMILIES = {
    "transition_metal_oxide_like",
    "phosphate_or_NASICON_like",
    "sulfate_like",
    "silicate_like",
    "prussian_blue_or_cyanide_like",
    "carbon_or_organic_like",
}


def classify_na_cathode_family(elements):
    if "Na" not in elements:
        return "not_sodium"

    if {"C", "N"}.issubset(elements) and len(elements & TRANSITION_METALS) > 0:
        return "prussian_blue_or_cyanide_like"

    if {"P", "O"}.issubset(elements):
        return "phosphate_or_NASICON_like"

    if {"S", "O"}.issubset(elements):
        return "sulfate_like"

    if {"Si", "O"}.issubset(elements):
        return "silicate_like"

    if "O" in elements and len(elements & TRANSITION_METALS) > 0:
        return "transition_metal_oxide_like"

    if "C" in elements:
        return "carbon_or_organic_like"

    return "other_or_unclear"


def element_flags(elements):
    return {
        "elements_clean": ",".join(sorted(elements)),
        "n_elements_clean": len(elements),

        "contains_Na": "Na" in elements,
        "contains_O": "O" in elements,
        "contains_P": "P" in elements,
        "contains_S": "S" in elements,
        "contains_C": "C" in elements,
        "contains_N": "N" in elements,
        "contains_H": "H" in elements,

        "contains_Fe": "Fe" in elements,
        "contains_Mn": "Mn" in elements,
        "contains_Ti": "Ti" in elements,
        "contains_V": "V" in elements,
        "contains_Cr": "Cr" in elements,
        "contains_Cu": "Cu" in elements,
        "contains_Co": "Co" in elements,
        "contains_Ni": "Ni" in elements,
        "contains_Li": "Li" in elements,

        "hard_exclusion_count": len(elements & HARD_EXCLUDE_ELEMENTS),
        "soft_penalty_count": len(elements & SOFT_PENALTY_ELEMENTS),
        "hard_exclusion_elements": ",".join(sorted(elements & HARD_EXCLUDE_ELEMENTS)),
        "soft_penalty_elements": ",".join(sorted(elements & SOFT_PENALTY_ELEMENTS)),
    }

In [6]:
metadata = {
    "download_datetime": datetime.now().isoformat(),
    "api_source": "Materials Project mp-api",
    "project_focus": "Sodium-ion battery cathode screening",
}

with MPRester(MP_API_KEY) as mpr:
    try:
        metadata["mp_database_version"] = mpr.get_database_version()
    except Exception as e:
        metadata["mp_database_version"] = None
        metadata["mp_database_version_error"] = str(e)

    try:
        metadata["summary_available_fields_count"] = len(mpr.materials.summary.available_fields)
        metadata["summary_available_fields_sample"] = list(mpr.materials.summary.available_fields[:40])
    except Exception as e:
        metadata["summary_available_fields_error"] = str(e)

metadata_path = META_DIR / "materials_project_download_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

C:\Users\IICT-1\AppData\Local\Temp\7\ipykernel_101328\749878477.py:9: MPRestWarning: `get_database_version` has been deprecated in favor of MPRester().db_version.
  metadata["mp_database_version"] = mpr.get_database_version()


{
  "download_datetime": "2026-06-21T19:20:58.457472",
  "api_source": "Materials Project mp-api",
  "project_focus": "Sodium-ion battery cathode screening",
  "mp_database_version": "2026.04.13",
  "summary_available_fields_count": 69,
  "summary_available_fields_sample": [
    "phonon_IDs",
    "theoretical",
    "database_IDs",
    "possible_species",
    "weighted_surface_energy_EV_PER_ANG2",
    "weighted_surface_energy",
    "weighted_work_function",
    "surface_anisotropy",
    "shape_factor",
    "has_reconstructed",
    "e_ij_max",
    "e_total",
    "e_ionic",
    "e_electronic",
    "n",
    "bulk_modulus",
    "shear_modulus",
    "universal_anisotropy",
    "homogeneous_poisson",
    "is_magnetic",
    "ordering",
    "total_magnetization",
    "total_magnetization_normalized_vol",
    "total_magnetization_normalized_formula_units",
    "num_magnetic_sites",
    "num_unique_magnetic_sites",
    "types_of_magnetic_species",
    "dos",
    "dos_energy_up",
    "dos_energy_d

In [7]:
print("Downloading Na-ion insertion electrode records...")

with MPRester(MP_API_KEY) as mpr:
    electrode_rester = getattr(mpr, "insertion_electrodes", None)

    if electrode_rester is None:
        electrode_rester = mpr.materials.insertion_electrodes

    na_electrode_docs = electrode_rester.search(
        working_ion="Na",
        all_fields=True,
        chunk_size=1000,
    )

na_electrode_records = [doc_to_dict(doc) for doc in na_electrode_docs]

df_na_electrodes_all = save_records_as_json_and_csv(
    na_electrode_records,
    "mp_na_insertion_electrodes_all_fields"
)

print("Total Na-ion electrode records:", len(df_na_electrodes_all))
display(df_na_electrodes_all.head())

C:\Users\IICT-1\AppData\Local\Temp\7\ipykernel_101328\680266061.py:4: DeprecationWarning: Accessing insertion_electrodes data through MPRester.insertion_electrodes is deprecated. Please use MPRester.materials.insertion_electrodes instead.
  electrode_rester = getattr(mpr, "insertion_electrodes", None)


Retrieving InsertionElectrodeDoc documents:   0%|          | 0/416 [00:00<?, ?it/s]

Saved JSON: mp_sodium_battery_dataset\raw\mp_na_insertion_electrodes_all_fields.json
Saved CSV : mp_sodium_battery_dataset\processed\mp_na_insertion_electrodes_all_fields.csv
Shape     : (416, 182)
Total Na-ion electrode records: 416


,formula_discharge,working_ion,material_ids,formula_charge,fracA_charge,nelements,max_delta_volume,framework_formula,energy_grav,num_steps,...,entries_composition_summary__all_composition_reduced__Cl,framework__Cl,entries_composition_summary__all_composition_reduced__Hf,framework__Hf,entries_composition_summary__all_composition_reduced__Pr,framework__Pr,entries_composition_summary__all_composition_reduced__Zr,framework__Zr,entries_composition_summary__all_composition_reduced__Te,framework__Te
0,Na3Ag,Na,"[""mp-aaaaaaeu"", ""mp-aaacpmmb""]",Ag,0.0,1,5.098846,Ag,-4.073349,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Na2AgAsF6,Na,"[""mp-aaaaccyz"", ""mp-aaaclfqj""]",AgAsF6,0.0,3,0.143002,AgAsF6,369.798324,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Na2TiAgF6,Na,"[""mp-aaaaapzu"", ""mp-aaaclfyx""]",TiAgF6,0.0,3,0.352928,TiAgF6,409.645624,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaMn3Ag(PO4)3,Na,"[""mp-aaacrnsr"", ""mp-aaacqwfh""]",Mn3Ag(PO4)3,0.0,4,0.037968,Mn3Ag(PO4)3,192.456646,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaAgO,Na,"[""mp-aaaaameg"", ""mp-aaacfkko""]",AgO,0.0,2,0.431291,AgO,362.558834,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_electrode_corrected = df_na_electrodes_all.copy()

parse_formulas = []

for _, row in df_electrode_corrected.iterrows():
    formula = row.get("formula_discharge", None)

    if pd.isna(formula) or str(formula).strip() == "":
        formula = row.get("formula_charge", None)

    if pd.isna(formula) or str(formula).strip() == "":
        formula = row.get("framework_formula", None)

    if pd.isna(formula) or str(formula).strip() == "":
        formula = row.get("battery_formula", None)

    parse_formulas.append(formula)

df_electrode_corrected["formula_for_parsing"] = parse_formulas

electrode_element_sets = df_electrode_corrected["formula_for_parsing"].map(elements_from_formula)

df_electrode_flags = pd.DataFrame(
    [element_flags(els) for els in electrode_element_sets]
)

df_electrode_corrected = pd.concat(
    [df_electrode_corrected.reset_index(drop=True), df_electrode_flags.reset_index(drop=True)],
    axis=1
)

df_electrode_corrected["preliminary_family"] = electrode_element_sets.map(classify_na_cathode_family)

corrected_path = CORRECTED_DIR / "mp_na_insertion_electrodes_corrected.csv"
make_csv_safe(df_electrode_corrected).to_csv(corrected_path, index=False)

print("Saved:", corrected_path)
print("Shape:", df_electrode_corrected.shape)

display(df_electrode_corrected["preliminary_family"].value_counts().to_frame("count"))
display(df_electrode_corrected.head())

Saved: mp_sodium_battery_dataset\corrected\mp_na_insertion_electrodes_corrected.csv
Shape: (416, 206)


,count
preliminary_family,
phosphate_or_NASICON_like,141
transition_metal_oxide_like,122
other_or_unclear,107
sulfate_like,23
silicate_like,18
carbon_or_organic_like,5


,formula_discharge,working_ion,material_ids,formula_charge,fracA_charge,nelements,max_delta_volume,framework_formula,energy_grav,num_steps,...,contains_Cr,contains_Cu,contains_Co,contains_Ni,contains_Li,hard_exclusion_count,soft_penalty_count,hard_exclusion_elements,soft_penalty_elements,preliminary_family
0,Na3Ag,Na,"[""mp-aaaaaaeu"", ""mp-aaacpmmb""]",Ag,0.0,1,5.098846,Ag,-4.073349,1,...,False,False,False,False,False,1,0,Ag,,other_or_unclear
1,Na2AgAsF6,Na,"[""mp-aaaaccyz"", ""mp-aaaclfqj""]",AgAsF6,0.0,3,0.143002,AgAsF6,369.798324,1,...,False,False,False,False,False,2,0,"Ag,As",,other_or_unclear
2,Na2TiAgF6,Na,"[""mp-aaaaapzu"", ""mp-aaaclfyx""]",TiAgF6,0.0,3,0.352928,TiAgF6,409.645624,1,...,False,False,False,False,False,1,0,Ag,,other_or_unclear
3,NaMn3Ag(PO4)3,Na,"[""mp-aaacrnsr"", ""mp-aaacqwfh""]",Mn3Ag(PO4)3,0.0,4,0.037968,Mn3Ag(PO4)3,192.456646,1,...,False,False,False,False,False,1,0,Ag,,phosphate_or_NASICON_like
4,NaAgO,Na,"[""mp-aaaaameg"", ""mp-aaacfkko""]",AgO,0.0,2,0.431291,AgO,362.558834,1,...,False,False,False,False,False,1,0,Ag,,other_or_unclear


In [9]:
df_screen = df_electrode_corrected.copy()

numeric_cols = [
    "average_voltage",
    "capacity_grav",
    "capacity_vol",
    "energy_grav",
    "energy_vol",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
]

for col in numeric_cols:
    if col not in df_screen.columns:
        df_screen[col] = np.nan
    df_screen[col] = pd.to_numeric(df_screen[col], errors="coerce")

df_screen["valid_voltage_window"] = df_screen["average_voltage"].between(2.0, 4.5, inclusive="both")
df_screen["valid_capacity"] = df_screen["capacity_grav"] >= 50
df_screen["valid_energy"] = df_screen["energy_grav"] >= 150
df_screen["valid_volume_change"] = df_screen["max_delta_volume"].fillna(999) <= 35

df_screen["valid_stability"] = (
    df_screen["stability_charge"].fillna(999) <= 0.10
) & (
    df_screen["stability_discharge"].fillna(999) <= 0.10
)

df_screen["family_relevant"] = df_screen["preliminary_family"].isin(ALLOWED_FAMILIES)

df_screen["basic_screening_pass"] = (
    df_screen["valid_voltage_window"]
    & df_screen["valid_capacity"]
    & df_screen["valid_energy"]
    & df_screen["valid_volume_change"]
    & df_screen["valid_stability"]
    & df_screen["family_relevant"]
)

df_screen["sustainability_strict_pass"] = df_screen["hard_exclusion_count"] == 0
df_screen["sustainability_moderate_pass"] = df_screen["hard_exclusion_count"] <= 1

df_screen["hard_exclusion_free_candidate"] = (
    df_screen["basic_screening_pass"]
    & df_screen["sustainability_strict_pass"]
)

df_screen["moderate_sustainable_candidate"] = (
    df_screen["basic_screening_pass"]
    & df_screen["sustainability_moderate_pass"]
)

screen_path = CORRECTED_DIR / "mp_na_electrodes_screened_candidates.csv"
make_csv_safe(df_screen).to_csv(screen_path, index=False)

print("Saved:", screen_path)
print("Total MP electrode records:", len(df_screen))
print("Basic screening pass:", int(df_screen["basic_screening_pass"].sum()))
print("Hard-exclusion-free candidates:", int(df_screen["hard_exclusion_free_candidate"].sum()))
print("Moderate sustainable candidates:", int(df_screen["moderate_sustainable_candidate"].sum()))

display(
    df_screen[
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "max_delta_volume",
            "stability_charge",
            "stability_discharge",
            "hard_exclusion_elements",
            "soft_penalty_elements",
            "basic_screening_pass",
            "hard_exclusion_free_candidate",
            "moderate_sustainable_candidate",
        ]
    ].head(20)
)

Saved: mp_sodium_battery_dataset\corrected\mp_na_electrodes_screened_candidates.csv
Total MP electrode records: 416
Basic screening pass: 164
Hard-exclusion-free candidates: 112
Moderate sustainable candidates: 162


,battery_formula,formula_discharge,framework_formula,preliminary_family,average_voltage,capacity_grav,energy_grav,max_delta_volume,stability_charge,stability_discharge,hard_exclusion_elements,soft_penalty_elements,basic_screening_pass,hard_exclusion_free_candidate,moderate_sustainable_candidate
0,Na0-3Ag,Na3Ag,Ag,other_or_unclear,-0.008959,454.679804,-4.073349,5.098846,0.003609,0.045182,Ag,,False,False,False
1,Na0-2AgAsF6,Na2AgAsF6,AgAsF6,other_or_unclear,2.364645,156.386393,369.798324,0.143002,0.000000,0.218388,"Ag,As",,False,False,False
2,Na0-2TiAgF6,Na2TiAgF6,TiAgF6,other_or_unclear,2.412688,169.788047,409.645624,0.352928,0.005566,0.395015,Ag,,False,False,False
3,Na0-1Mn3Ag(PO4)3,NaMn3Ag(PO4)3,Mn3Ag(PO4)3,phosphate_or_NASICON_like,4.169086,46.162795,192.456646,0.037968,0.036993,0.000000,Ag,,False,False,False
4,Na0-1AgO,NaAgO,AgO,other_or_unclear,1.986623,182.500077,362.558834,0.431291,0.308693,0.291835,Ag,,False,False,False
5,Na2-3BiBAsO7,Na3BiBAsO7,BiBAsO7,other_or_unclear,3.620409,56.343735,203.987359,0.030114,0.068633,0.022577,As,Bi,False,False,False
6,Na2-3CoBAsO7,Na3CoBAsO7,CoBAsO7,transition_metal_oxide_like,3.710628,82.306320,305.408175,0.031366,0.113283,0.077535,"As,Co",,False,False,False
7,Na2-3FeBAsO7,Na3FeBAsO7,FeBAsO7,transition_metal_oxide_like,3.612906,83.094364,300.212088,0.025283,0.055859,0.030033,As,,True,False,True
8,Na2-3MnBAsO7,Na3MnBAsO7,MnBAsO7,transition_metal_oxide_like,3.503137,83.328675,291.911782,0.026737,0.055802,0.016847,As,,True,False,True
9,Na2-3VBAsO7,Na3VBAsO7,VBAsO7,transition_metal_oxide_like,2.894382,84.377119,244.219584,0.031270,0.082883,0.021307,As,V,True,False,True


In [10]:
def clip01(x):
    return np.clip(x, 0, 1)


df_ranked = df_screen.copy()

df_ranked["voltage_score"] = clip01((df_ranked["average_voltage"] - 2.0) / (4.5 - 2.0))
df_ranked["capacity_score"] = clip01(df_ranked["capacity_grav"] / 200)
df_ranked["energy_score"] = clip01(df_ranked["energy_grav"] / 600)

df_ranked["volume_score"] = clip01(
    1 - (df_ranked["max_delta_volume"].fillna(40) / 40)
)

worst_stability = df_ranked[["stability_charge", "stability_discharge"]].max(axis=1)

df_ranked["stability_score"] = clip01(
    1 - (worst_stability.fillna(0.2) / 0.2)
)

df_ranked["criticality_score"] = clip01(
    1
    - 0.25 * df_ranked["hard_exclusion_count"]
    - 0.08 * df_ranked["soft_penalty_count"]
)

df_ranked["mp_preliminary_score"] = (
    0.25 * df_ranked["voltage_score"]
    + 0.25 * df_ranked["capacity_score"]
    + 0.20 * df_ranked["energy_score"]
    + 0.15 * df_ranked["stability_score"]
    + 0.05 * df_ranked["volume_score"]
    + 0.10 * df_ranked["criticality_score"]
)

df_ranked = df_ranked.sort_values(
    by=["hard_exclusion_free_candidate", "mp_preliminary_score"],
    ascending=[False, False]
)

ranked_path = CORRECTED_DIR / "mp_na_electrodes_ranked_preliminary.csv"
make_csv_safe(df_ranked).to_csv(ranked_path, index=False)

print("Saved:", ranked_path)

display(
    df_ranked[
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "max_delta_volume",
            "stability_charge",
            "stability_discharge",
            "hard_exclusion_elements",
            "soft_penalty_elements",
            "hard_exclusion_free_candidate",
            "mp_preliminary_score",
        ]
    ].head(30)
)

Saved: mp_sodium_battery_dataset\corrected\mp_na_electrodes_ranked_preliminary.csv


,battery_formula,formula_discharge,framework_formula,preliminary_family,average_voltage,capacity_grav,energy_grav,max_delta_volume,stability_charge,stability_discharge,hard_exclusion_elements,soft_penalty_elements,hard_exclusion_free_candidate,mp_preliminary_score
163,Na0-2MnH4(SO5)2,Na2MnH4(SO5)2,MnH4(SO5)2,sulfate_like,4.239598,162.890624,690.590710,0.133276,0.018711,0.006873,,,True,0.913373
311,Na0-2MnPO4F,Na2MnPO4F,MnPO4F,phosphate_or_NASICON_like,3.864973,249.446804,964.105080,0.286562,0.035308,0.000000,,,True,0.909658
189,Na0-2MnP2O7,Na2MnP2O7,MnP2O7,phosphate_or_NASICON_like,4.058779,195.018501,791.536945,0.123101,0.072623,0.000365,,,True,0.895030
119,Na0-3Cr2(PO4)3,Na3Cr2(PO4)3,Cr2(PO4)3,phosphate_or_NASICON_like,4.291305,175.603252,753.567058,0.087594,0.061562,0.000000,,Cr,True,0.894354
93,Na0-2MnCSO7,Na2MnCSO7,MnCSO7,sulfate_like,4.100875,208.580697,855.363301,0.212925,0.087317,0.036720,,,True,0.894334
89,Na1-3MnPCO7,Na3MnPCO7,MnPCO7,phosphate_or_NASICON_like,3.369558,192.202735,647.638332,0.073266,0.000000,0.000000,,,True,0.877118
338,Na0-4Fe2(PO4)3,Na4Fe2(PO4)3,Fe2(PO4)3,phosphate_or_NASICON_like,3.956552,219.431043,868.190282,0.036937,0.096099,0.018703,,,True,0.873535
78,Na0-2FeCSO7,Na2FeCSO7,FeCSO7,sulfate_like,3.892605,207.847172,809.067032,0.128771,0.089343,0.035865,,,True,0.872092
120,Na0-3.5Cr2(PO4)3,Na7Cr4(PO4)6,Cr2(PO4)3,phosphate_or_NASICON_like,3.702667,199.853183,739.989847,0.162168,0.062883,0.037110,,Cr,True,0.864718
77,Na2-6Fe2C4SO16,Na6Fe2C4SO16,Fe2C4SO16,sulfate_like,3.904088,183.030590,714.567591,0.034428,0.083437,0.000000,,,True,0.856576


In [11]:
linked_mpids = set()

for col in ["id_charge", "id_discharge"]:
    if col in df_electrode_corrected.columns:
        linked_mpids.update(
            df_electrode_corrected[col]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )

linked_mpids = sorted([x for x in linked_mpids if x.startswith("mp-")])

print("Unique linked MP material IDs:", len(linked_mpids))
print(linked_mpids[:20])

Unique linked MP material IDs: 832
['mp-aaaaaaax', 'mp-aaaaaabt', 'mp-aaaaaadc', 'mp-aaaaaady', 'mp-aaaaaaeu', 'mp-aaaaaahi', 'mp-aaaaaajm', 'mp-aaaaaavp', 'mp-aaaaabet', 'mp-aaaaadey', 'mp-aaaaadgx', 'mp-aaaaaeye', 'mp-aaaaafoa', 'mp-aaaaafxc', 'mp-aaaaagsr', 'mp-aaaaaibp', 'mp-aaaaaivb', 'mp-aaaaajfm', 'mp-aaaaajor', 'mp-aaaaajot']


In [12]:
summary_fields_requested = [
    "material_id",
    "formula_pretty",
    "composition_reduced",
    "chemsys",
    "elements",
    "nelements",
    "nsites",
    "volume",
    "density",
    "density_atomic",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "is_stable",
    "theoretical",
    "band_gap",
    "is_gap_direct",
    "is_metal",
    "total_magnetization",
    "last_updated",
]

linked_summary_docs = []

with MPRester(MP_API_KEY) as mpr:
    available_fields = set(mpr.materials.summary.available_fields)

    summary_fields = [
        f for f in summary_fields_requested
        if f in available_fields
    ]

    print("Using summary fields:", summary_fields)

    for i in tqdm(range(0, len(linked_mpids), 500), desc="Downloading linked summaries"):
        batch = linked_mpids[i:i + 500]

        docs = mpr.materials.summary.search(
            material_ids=batch,
            fields=summary_fields,
        )

        linked_summary_docs.extend(docs)
        time.sleep(0.2)

linked_summary_records = [doc_to_dict(doc) for doc in linked_summary_docs]

df_linked_summary_corrected = pd.json_normalize(linked_summary_records, sep="__")
df_linked_summary_corrected = make_csv_safe(df_linked_summary_corrected)

linked_summary_path = CORRECTED_DIR / "mp_na_electrode_linked_materials_summary_corrected.csv"
df_linked_summary_corrected.to_csv(linked_summary_path, index=False)

print("Saved:", linked_summary_path)
print("Shape:", df_linked_summary_corrected.shape)

display(df_linked_summary_corrected.head())

Using summary fields: ['material_id', 'formula_pretty', 'composition_reduced', 'chemsys', 'elements', 'nelements', 'nsites', 'volume', 'density', 'density_atomic', 'symmetry', 'energy_above_hull', 'formation_energy_per_atom', 'is_stable', 'theoretical', 'band_gap', 'is_gap_direct', 'is_metal', 'total_magnetization', 'last_updated']


Retrieving SummaryDoc documents:   0%|          | 0/498 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/331 [00:00<?, ?it/s]

Saved: mp_sodium_battery_dataset\corrected\mp_na_electrode_linked_materials_summary_corrected.csv
Shape: (829, 79)


,nsites,formula_pretty,nelements,density_atomic,theoretical,volume,is_metal,is_stable,chemsys,total_magnetization,...,composition_reduced__Tl,composition_reduced__Nd,composition_reduced__In,composition_reduced__Mg,composition_reduced__Cd,composition_reduced__N,composition_reduced__Sc,composition_reduced__Rh,composition_reduced__Hf,composition_reduced__Zr
0,1,Ni,1,10.492020,False,10.492020,True,True,Ni,7.073310e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Ca,1,43.360964,False,43.360964,True,True,Ca,2.600000e-06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Sb,1,26.654143,False,53.308285,True,False,Sb,2.000000e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Co,1,10.841543,False,10.841543,True,True,Co,1.838000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Ag,1,17.285231,False,17.285231,True,False,Ag,9.000000e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
print("Downloading broad Na-containing Materials Project summary data...")

# Important:
# Do NOT pass the long HARD_EXCLUDE_ELEMENTS list to exclude_elements.
# MP API has a length limit for exclude_elements.
# We download all Na-containing materials and filter locally in Cell 14.

summary_fields_requested = [
    "material_id",
    "formula_pretty",
    "composition_reduced",
    "chemsys",
    "elements",
    "nelements",
    "nsites",
    "volume",
    "density",
    "density_atomic",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "is_stable",
    "theoretical",
    "band_gap",
    "is_gap_direct",
    "is_metal",
    "total_magnetization",
    "last_updated",
]

broad_summary_docs = []

with MPRester(MP_API_KEY) as mpr:
    available_fields = set(mpr.materials.summary.available_fields)

    summary_fields = [
        f for f in summary_fields_requested
        if f in available_fields
    ]

    print("Using summary fields:")
    print(summary_fields)

    # Robust and reviewer-safe method:
    # Download all Na-containing materials, then apply all exclusion rules locally.
    broad_summary_docs = mpr.materials.summary.search(
        elements=["Na"],
        fields=summary_fields,
        chunk_size=1000,
    )

broad_summary_records = [doc_to_dict(doc) for doc in broad_summary_docs]

df_broad = pd.json_normalize(broad_summary_records, sep="__")
df_broad = make_csv_safe(df_broad)

broad_raw_path = PROCESSED_DIR / "mp_all_na_containing_materials_summary_raw.csv"
df_broad.to_csv(broad_raw_path, index=False)

print("Saved:", broad_raw_path)
print("Shape:", df_broad.shape)

display(df_broad.head())

Using summary fields:
['material_id', 'formula_pretty', 'composition_reduced', 'chemsys', 'elements', 'nelements', 'nsites', 'volume', 'density', 'density_atomic', 'symmetry', 'energy_above_hull', 'formation_energy_per_atom', 'is_stable', 'theoretical', 'band_gap', 'is_gap_direct', 'is_metal', 'total_magnetization', 'last_updated']


Retrieving SummaryDoc documents:   0%|          | 0/12834 [00:00<?, ?it/s]

Saved: mp_sodium_battery_dataset\processed\mp_all_na_containing_materials_summary_raw.csv
Shape: (12834, 113)


,nsites,formula_pretty,nelements,density_atomic,theoretical,volume,is_metal,is_stable,chemsys,total_magnetization,...,composition_reduced__Tc,composition_reduced__Lu,composition_reduced__Os,composition_reduced__Br,composition_reduced__U,composition_reduced__Ir,composition_reduced__Rh,composition_reduced__Np,composition_reduced__Th,composition_reduced__Xe
0,20,NaLi4Cl5,3,17.639988,True,352.799753,False,False,Cl-Li-Na,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30,NaMgCl3,3,22.645550,True,679.366486,False,False,Cl-Mg-Na,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,40,NaNO3,3,11.986894,True,479.475779,False,False,N-Na-O,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30,NaH9O5,3,9.150288,True,274.508640,False,False,H-Na-O,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20,NaLi4Cl5,3,17.601156,True,352.023121,False,False,Cl-Li-Na,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
if "chemsys" not in df_broad.columns:
    raise ValueError("chemsys column is missing. Cannot apply robust element filtering.")

broad_element_sets = df_broad["chemsys"].map(elements_from_chemsys)

df_broad_flags = pd.DataFrame(
    [element_flags(els) for els in broad_element_sets]
)

df_broad_corrected = pd.concat(
    [df_broad.reset_index(drop=True), df_broad_flags.reset_index(drop=True)],
    axis=1
)

df_broad_corrected["preliminary_family"] = broad_element_sets.map(classify_na_cathode_family)
df_broad_corrected["sustainability_strict_pass"] = df_broad_corrected["hard_exclusion_count"] == 0
df_broad_corrected["family_relevant"] = df_broad_corrected["preliminary_family"].isin(ALLOWED_FAMILIES)

if "energy_above_hull" in df_broad_corrected.columns:
    df_broad_corrected["energy_above_hull"] = pd.to_numeric(
        df_broad_corrected["energy_above_hull"],
        errors="coerce",
    )
    df_broad_corrected["near_hull_50mev"] = df_broad_corrected["energy_above_hull"] <= 0.05
else:
    df_broad_corrected["near_hull_50mev"] = False

df_broad_candidates = df_broad_corrected[
    df_broad_corrected["contains_Na"]
    & df_broad_corrected["family_relevant"]
    & df_broad_corrected["sustainability_strict_pass"]
    & df_broad_corrected["near_hull_50mev"]
].copy()

broad_corrected_path = CORRECTED_DIR / "mp_all_na_containing_summary_corrected_with_flags.csv"
broad_candidates_path = CORRECTED_DIR / "mp_broad_na_candidate_pool_strict_near_hull.csv"

make_csv_safe(df_broad_corrected).to_csv(broad_corrected_path, index=False)
make_csv_safe(df_broad_candidates).to_csv(broad_candidates_path, index=False)

print("Saved:", broad_corrected_path)
print("Saved:", broad_candidates_path)

print("Broad Na records:", len(df_broad_corrected))
print("Strict near-hull broad pool:", len(df_broad_candidates))

print("\nBroad family counts:")
display(df_broad_corrected["preliminary_family"].value_counts().to_frame("count"))

print("\nStrict near-hull broad pool family counts:")
display(df_broad_candidates["preliminary_family"].value_counts().to_frame("count"))

display(df_broad_candidates.head(20))

C:\Users\IICT-1\AppData\Local\Temp\7\ipykernel_101328\2815505383.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_broad_corrected["preliminary_family"] = broad_element_sets.map(classify_na_cathode_family)
C:\Users\IICT-1\AppData\Local\Temp\7\ipykernel_101328\2815505383.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_broad_corrected["sustainability_strict_pass"] = df_broad_corrected["hard_exclusion_count"] == 0
C:\Users\IICT-1\AppData\Local\Temp\7\ipykernel_101328\2815505383.py:17: PerformanceWarning: DataFrame is 

Saved: mp_sodium_battery_dataset\corrected\mp_all_na_containing_summary_corrected_with_flags.csv
Saved: mp_sodium_battery_dataset\corrected\mp_broad_na_candidate_pool_strict_near_hull.csv
Broad Na records: 12834
Strict near-hull broad pool: 3544

Broad family counts:


,count
preliminary_family,
other_or_unclear,4255
phosphate_or_NASICON_like,3031
sulfate_like,2239
transition_metal_oxide_like,2182
silicate_like,860
carbon_or_organic_like,238
prussian_blue_or_cyanide_like,29



Strict near-hull broad pool family counts:


,count
preliminary_family,
phosphate_or_NASICON_like,1753
transition_metal_oxide_like,726
sulfate_like,500
silicate_like,493
carbon_or_organic_like,72


,nsites,formula_pretty,nelements,density_atomic,theoretical,volume,is_metal,is_stable,chemsys,total_magnetization,...,contains_Ni,contains_Li,hard_exclusion_count,soft_penalty_count,hard_exclusion_elements,soft_penalty_elements,preliminary_family,sustainability_strict_pass,family_relevant,near_hull_50mev
55,10,NaSrYWO6,5,14.523411,True,145.234108,False,False,Na-O-Sr-W-Y,0.000000,...,False,False,0,2,,"W,Y",transition_metal_oxide_like,True,True,True
66,40,NaSrPrWO6,5,15.111826,True,604.473041,False,False,Na-O-Pr-Sr-W,0.000000,...,False,False,0,2,,"Pr,W",transition_metal_oxide_like,True,True,True
109,40,NaCaFe2(SiO3)4,5,11.453509,True,458.140347,False,False,Ca-Fe-Na-O-Si,7.999998,...,False,False,0,0,,,silicate_like,True,True,True
115,52,NaV3H6(SO7)2,5,10.439218,True,542.839356,False,False,H-Na-O-S-V,0.000203,...,False,False,0,1,,V,sulfate_like,True,True,True
116,13,NaMn4O8,3,12.015741,True,156.204629,True,False,Mn-Na-O,11.002914,...,False,False,0,0,,,transition_metal_oxide_like,True,True,True
214,12,NaCu2O3,3,11.038287,True,132.459446,True,False,Cu-Na-O,0.120652,...,False,False,0,0,,,transition_metal_oxide_like,True,True,True
231,108,NaZrGaP2SO12,6,14.025037,True,1514.703966,False,False,Ga-Na-O-P-S-Zr,0.000002,...,False,False,0,0,,,phosphate_or_NASICON_like,True,True,True
233,108,NaMgNbP2SO12,6,13.983874,True,1510.258390,False,False,Mg-Na-Nb-O-P-S,0.000408,...,False,False,0,0,,,phosphate_or_NASICON_like,True,True,True
238,108,NaAlGeP2SO12,6,12.122242,True,1309.202170,False,False,Al-Ge-Na-O-P-S,0.000810,...,False,False,0,0,,,phosphate_or_NASICON_like,True,True,True
242,108,NaAlSnP2SO12,6,13.164069,True,1421.719402,False,False,Al-Na-O-P-S-Sn,0.000003,...,False,False,0,0,,,phosphate_or_NASICON_like,True,True,True


In [15]:
df_final = df_ranked.copy()

df_final["element_set_v2"] = df_final["elements_clean"].map(element_set_from_string)

df_final["hard_exclusion_v2_elements"] = df_final["element_set_v2"].map(
    lambda s: ",".join(sorted(s & HARD_EXCLUDE_V2))
)

df_final["soft_penalty_v2_elements"] = df_final["element_set_v2"].map(
    lambda s: ",".join(sorted(s & SOFT_PENALTY_V2))
)

df_final["hard_exclusion_v2_count"] = df_final["hard_exclusion_v2_elements"].map(
    lambda x: 0 if x == "" else len(x.split(","))
)

df_final["soft_penalty_v2_count"] = df_final["soft_penalty_v2_elements"].map(
    lambda x: 0 if x == "" else len(x.split(","))
)

df_final["conservative_earth_abundant_candidate"] = (
    df_final["basic_screening_pass"]
    & (df_final["hard_exclusion_v2_count"] == 0)
    & (df_final["soft_penalty_v2_count"] == 0)
)

df_final = df_final.sort_values(
    by=["conservative_earth_abundant_candidate", "hard_exclusion_free_candidate", "mp_preliminary_score"],
    ascending=[False, False, False],
)

final_path = FINAL_DIR / "mp_na_electrodes_conservative_candidates_v1.csv"
make_csv_safe(df_final).to_csv(final_path, index=False)

print("Saved:", final_path)
print("Total MP electrode records:", len(df_final))
print("Hard-exclusion-free candidates:", int(df_final["hard_exclusion_free_candidate"].sum()))
print("Conservative earth-abundant candidates:", int(df_final["conservative_earth_abundant_candidate"].sum()))

print("\nConservative candidate family counts:")
display(
    df_final[df_final["conservative_earth_abundant_candidate"]]
    ["preliminary_family"]
    .value_counts()
    .to_frame("count")
)

display(
    df_final[df_final["conservative_earth_abundant_candidate"]][
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "max_delta_volume",
            "stability_charge",
            "stability_discharge",
            "hard_exclusion_v2_elements",
            "soft_penalty_v2_elements",
            "mp_preliminary_score",
        ]
    ].head(40)
)

Saved: mp_sodium_battery_dataset\final\mp_na_electrodes_conservative_candidates_v1.csv
Total MP electrode records: 416
Hard-exclusion-free candidates: 112
Conservative earth-abundant candidates: 56

Conservative candidate family counts:


,count
preliminary_family,
phosphate_or_NASICON_like,34
transition_metal_oxide_like,10
sulfate_like,9
silicate_like,3


,battery_formula,formula_discharge,framework_formula,preliminary_family,average_voltage,capacity_grav,energy_grav,max_delta_volume,stability_charge,stability_discharge,hard_exclusion_v2_elements,soft_penalty_v2_elements,mp_preliminary_score
311,Na0-2MnPO4F,Na2MnPO4F,MnPO4F,phosphate_or_NASICON_like,3.864973,249.446804,964.105080,0.286562,0.035308,0.000000,,,0.909658
189,Na0-2MnP2O7,Na2MnP2O7,MnP2O7,phosphate_or_NASICON_like,4.058779,195.018501,791.536945,0.123101,0.072623,0.000365,,,0.895030
93,Na0-2MnCSO7,Na2MnCSO7,MnCSO7,sulfate_like,4.100875,208.580697,855.363301,0.212925,0.087317,0.036720,,,0.894334
89,Na1-3MnPCO7,Na3MnPCO7,MnPCO7,phosphate_or_NASICON_like,3.369558,192.202735,647.638332,0.073266,0.000000,0.000000,,,0.877118
338,Na0-4Fe2(PO4)3,Na4Fe2(PO4)3,Fe2(PO4)3,phosphate_or_NASICON_like,3.956552,219.431043,868.190282,0.036937,0.096099,0.018703,,,0.873535
78,Na0-2FeCSO7,Na2FeCSO7,FeCSO7,sulfate_like,3.892605,207.847172,809.067032,0.128771,0.089343,0.035865,,,0.872092
77,Na2-6Fe2C4SO16,Na6Fe2C4SO16,Fe2C4SO16,sulfate_like,3.904088,183.030590,714.567591,0.034428,0.083437,0.000000,,,0.856576
73,Na1-3FePCO7,Na3FePCO7,FePCO7,phosphate_or_NASICON_like,3.443900,191.579709,659.781296,0.010975,0.040622,0.000000,,,0.853385
150,Na0-1CuPO4,NaCuPO4,CuPO4,phosphate_or_NASICON_like,3.827827,147.660761,565.219775,0.119355,0.005488,0.027283,,,0.835154
190,Na0-6Mn3(PO4)4,Na6Mn3(PO4)4,Mn3(PO4)4,phosphate_or_NASICON_like,3.469876,235.569716,817.397652,0.140243,0.094573,0.085801,,,0.825882


In [16]:
expected_files = [
    RAW_DIR / "mp_na_insertion_electrodes_all_fields.json",
    PROCESSED_DIR / "mp_na_insertion_electrodes_all_fields.csv",
    META_DIR / "materials_project_download_metadata.json",
    CORRECTED_DIR / "mp_na_insertion_electrodes_corrected.csv",
    CORRECTED_DIR / "mp_na_electrodes_screened_candidates.csv",
    CORRECTED_DIR / "mp_na_electrodes_ranked_preliminary.csv",
    CORRECTED_DIR / "mp_na_electrode_linked_materials_summary_corrected.csv",
    CORRECTED_DIR / "mp_all_na_containing_summary_corrected_with_flags.csv",
    CORRECTED_DIR / "mp_broad_na_candidate_pool_strict_near_hull.csv",
    FINAL_DIR / "mp_na_electrodes_conservative_candidates_v1.csv",
]

print("Final file audit:")

for path in expected_files:
    print(f"{'FOUND' if path.exists() else 'MISSING'} - {path}")

print("\nMain final dataset:")
display(df_final.head())

print("\nMain conservative candidate subset:")
display(df_final[df_final["conservative_earth_abundant_candidate"]].head())

Final file audit:
FOUND - mp_sodium_battery_dataset\raw\mp_na_insertion_electrodes_all_fields.json
FOUND - mp_sodium_battery_dataset\processed\mp_na_insertion_electrodes_all_fields.csv
FOUND - mp_sodium_battery_dataset\metadata\materials_project_download_metadata.json
FOUND - mp_sodium_battery_dataset\corrected\mp_na_insertion_electrodes_corrected.csv
FOUND - mp_sodium_battery_dataset\corrected\mp_na_electrodes_screened_candidates.csv
FOUND - mp_sodium_battery_dataset\corrected\mp_na_electrodes_ranked_preliminary.csv
FOUND - mp_sodium_battery_dataset\corrected\mp_na_electrode_linked_materials_summary_corrected.csv
FOUND - mp_sodium_battery_dataset\corrected\mp_all_na_containing_summary_corrected_with_flags.csv
FOUND - mp_sodium_battery_dataset\corrected\mp_broad_na_candidate_pool_strict_near_hull.csv
FOUND - mp_sodium_battery_dataset\final\mp_na_electrodes_conservative_candidates_v1.csv

Main final dataset:


,formula_discharge,working_ion,material_ids,formula_charge,fracA_charge,nelements,max_delta_volume,framework_formula,energy_grav,num_steps,...,volume_score,stability_score,criticality_score,mp_preliminary_score,element_set_v2,hard_exclusion_v2_elements,soft_penalty_v2_elements,hard_exclusion_v2_count,soft_penalty_v2_count,conservative_earth_abundant_candidate
311,Na2MnPO4F,Na,"[""mp-aaabxacc"", ""mp-aaabfrce""]",MnPO4F,0.000000,4,0.286562,MnPO4F,964.105080,1,...,0.992836,0.823462,1.0,0.909658,"{O, F, P, Na, Mn}",,,0,0,True
189,Na2MnP2O7,Na,"[""mp-aaabhhug"", ""mp-aaacbgbk"", ""mp-aaacqeli""]",MnP2O7,0.000000,3,0.123101,MnP2O7,791.536945,1,...,0.996922,0.636887,1.0,0.895030,"{O, P, Mn, Na}",,,0,0,True
93,Na2MnCSO7,Na,"[""mp-aaabqyiz"", ""mp-aaabrxto""]",MnCSO7,0.000000,4,0.212925,MnCSO7,855.363301,1,...,0.994677,0.563417,1.0,0.894334,"{O, S, Na, Mn, C}",,,0,0,True
89,Na3MnPCO7,Na,"[""mp-aaabrhtw"", ""mp-aaabrsfg"", ""mp-aaabsfnd"", ...",NaMnPCO7,0.090909,4,0.073266,MnPCO7,647.638332,2,...,0.998168,1.000000,1.0,0.877118,"{O, P, Na, Mn, C}",,,0,0,True
338,Na4Fe2(PO4)3,Na,"[""mp-aaaabnik"", ""mp-aaaabnnc"", ""mp-aaaabllc"", ...",Fe2(PO4)3,0.000000,3,0.036937,Fe2(PO4)3,868.190282,2,...,0.999077,0.519503,1.0,0.873535,"{O, P, Na, Fe}",,,0,0,True



Main conservative candidate subset:


,formula_discharge,working_ion,material_ids,formula_charge,fracA_charge,nelements,max_delta_volume,framework_formula,energy_grav,num_steps,...,volume_score,stability_score,criticality_score,mp_preliminary_score,element_set_v2,hard_exclusion_v2_elements,soft_penalty_v2_elements,hard_exclusion_v2_count,soft_penalty_v2_count,conservative_earth_abundant_candidate
311,Na2MnPO4F,Na,"[""mp-aaabxacc"", ""mp-aaabfrce""]",MnPO4F,0.000000,4,0.286562,MnPO4F,964.105080,1,...,0.992836,0.823462,1.0,0.909658,"{O, F, P, Na, Mn}",,,0,0,True
189,Na2MnP2O7,Na,"[""mp-aaabhhug"", ""mp-aaacbgbk"", ""mp-aaacqeli""]",MnP2O7,0.000000,3,0.123101,MnP2O7,791.536945,1,...,0.996922,0.636887,1.0,0.895030,"{O, P, Mn, Na}",,,0,0,True
93,Na2MnCSO7,Na,"[""mp-aaabqyiz"", ""mp-aaabrxto""]",MnCSO7,0.000000,4,0.212925,MnCSO7,855.363301,1,...,0.994677,0.563417,1.0,0.894334,"{O, S, Na, Mn, C}",,,0,0,True
89,Na3MnPCO7,Na,"[""mp-aaabrhtw"", ""mp-aaabrsfg"", ""mp-aaabsfnd"", ...",NaMnPCO7,0.090909,4,0.073266,MnPCO7,647.638332,2,...,0.998168,1.000000,1.0,0.877118,"{O, P, Na, Mn, C}",,,0,0,True
338,Na4Fe2(PO4)3,Na,"[""mp-aaaabnik"", ""mp-aaaabnnc"", ""mp-aaaabllc"", ...",Fe2(PO4)3,0.000000,3,0.036937,Fe2(PO4)3,868.190282,2,...,0.999077,0.519503,1.0,0.873535,"{O, P, Na, Fe}",,,0,0,True
